In [1]:
"""
check_missing.py
----------------
Scans every case folder under BASE_DIR and reports:

  1. Folders with NO raw CSV frames at all
     (averaging was never possible)

  2. Folders where the time_average CSV is MISSING
     (averaging was not completed / failed)

  3. Folders where BOTH raw frames AND the time_average CSV exist
     (everything looks good)

Usage (Jupyter Notebook)
------------------------
    %run check_missing.py
"""

import os
import re

# ── Configuration ─────────────────────────────────────────────────────────────
BASE_DIR      = r"D:\FCAI\effusion_coupon_CompundAngle\infared_data"

OUTPUT_SUBDIR = "time_average"       # sub-folder created by compute_time_average.py
OUTPUT_CSV    = "time_average.csv"   # expected output file name

PLOT_SUBDIR   = "time_averaged_plot_intensity"  # skip this folder if present at top level
# ──────────────────────────────────────────────────────────────────────────────

FRAME_RE = re.compile(r"^.+_\d+\.csv$", re.IGNORECASE)


def has_raw_frames(case_dir):
    """Return True if the folder contains at least one frame CSV (e.g. Rec-0006_0.csv)."""
    try:
        files = os.listdir(case_dir)
    except PermissionError:
        return False
    return any(FRAME_RE.match(f) for f in files)


def count_raw_frames(case_dir):
    """Return the number of frame CSVs found in case_dir."""
    try:
        files = os.listdir(case_dir)
    except PermissionError:
        return 0
    return sum(1 for f in files if FRAME_RE.match(f))


def has_time_average(case_dir):
    """Return True if the time_average CSV was successfully created."""
    avg_path = os.path.join(case_dir, OUTPUT_SUBDIR, OUTPUT_CSV)
    return os.path.isfile(avg_path)


def main():
    base = os.path.normpath(BASE_DIR)
    print("=" * 60)
    print("  Time-Average Completeness Check")
    print("=" * 60)
    print(f"  Base directory: {base}\n")

    if not os.path.isdir(base):
        print(f"ERROR: BASE_DIR does not exist or is not accessible:\n  {base}")
        print("  → Edit the BASE_DIR variable at the top of this script.")
        return

    entries = sorted(os.listdir(base))
    case_dirs = [
        os.path.join(base, e) for e in entries
        if os.path.isdir(os.path.join(base, e))
           and e not in (OUTPUT_SUBDIR, PLOT_SUBDIR)
    ]

    if not case_dirs:
        print("No case sub-folders found.")
        return

    print(f"  Total case folders found: {len(case_dirs)}\n")

    no_frames       = []   # no raw CSVs at all
    missing_average = []   # has raw CSVs but no time_average output
    complete        = []   # has both raw CSVs and time_average output

    for case_dir in case_dirs:
        name        = os.path.basename(case_dir)
        n_frames    = count_raw_frames(case_dir)
        has_avg     = has_time_average(case_dir)

        if n_frames == 0:
            no_frames.append(name)
        elif not has_avg:
            missing_average.append((name, n_frames))
        else:
            complete.append((name, n_frames))

    # ── Report ────────────────────────────────────────────────────────────────

    # 1. Folders with no raw frames
    print("─" * 60)
    print(f"[1] Folders with NO raw CSV frames ({len(no_frames)} found)")
    print("─" * 60)
    if no_frames:
        for name in no_frames:
            print(f"    ✗  {name}")
    else:
        print("    (none)")

    # 2. Folders missing time_average output
    print()
    print("─" * 60)
    print(f"[2] Folders with raw frames but MISSING time_average.csv ({len(missing_average)} found)")
    print("─" * 60)
    if missing_average:
        for name, n in missing_average:
            print(f"    ✗  {name}   ({n} frames present)")
    else:
        print("    (none)")

    # 3. Successfully completed
    print()
    print("─" * 60)
    print(f"[3] Folders with averaging COMPLETE ({len(complete)} found)")
    print("─" * 60)
    if complete:
        for name, n in complete:
            print(f"    ✓  {name}   ({n} frames averaged)")
    else:
        print("    (none)")

    # ── Summary ───────────────────────────────────────────────────────────────
    print()
    print("=" * 60)
    print("  SUMMARY")
    print("=" * 60)
    print(f"  Total cases         : {len(case_dirs)}")
    print(f"  Complete            : {len(complete)}")
    print(f"  Missing average     : {len(missing_average)}")
    print(f"  No raw frames       : {len(no_frames)}")
    needs_action = len(missing_average) + len(no_frames)
    if needs_action == 0:
        print("\n  ✓ All cases processed successfully!")
    else:
        print(f"\n  ✗ {needs_action} case(s) need attention.")
    print("=" * 60)


main()

  Time-Average Completeness Check
  Base directory: D:\FCAI\effusion_coupon_CompundAngle\infared_data

  Total case folders found: 22

────────────────────────────────────────────────────────────
[1] Folders with NO raw CSV frames (4 found)
────────────────────────────────────────────────────────────
    ✗  .ipynb_checkpoints
    ✗  anaconda_projects
    ✗  cr_375_phi_137
    ✗  cr_500_phi_137

────────────────────────────────────────────────────────────
[2] Folders with raw frames but MISSING time_average.csv (0 found)
────────────────────────────────────────────────────────────
    (none)

────────────────────────────────────────────────────────────
[3] Folders with averaging COMPLETE (18 found)
────────────────────────────────────────────────────────────
    ✓  cr_1125_phi_09   (500 frames averaged)
    ✓  cr_1125_phi_10   (500 frames averaged)
    ✓  cr_1125_phi_118   (500 frames averaged)
    ✓  cr_1380_phi_09   (500 frames averaged)
    ✓  cr_1380_phi_10   (500 frames averaged)
 